In [4]:
import kagglehub

tomato_path = kagglehub.dataset_download(
    "yusufmurtaza01/tomato-leaf-disease"
)

print(tomato_path)

Using Colab cache for faster access to the 'tomato-leaf-disease' dataset.
/kaggle/input/tomato-leaf-disease


In [6]:
TOMATO_ROOT = "/kaggle/input/tomato-leaf-disease"

In [7]:
import os

for root, dirs, files in os.walk(TOMATO_ROOT):
    print(root)

/kaggle/input/tomato-leaf-disease
/kaggle/input/tomato-leaf-disease/tomato
/kaggle/input/tomato-leaf-disease/tomato/labels
/kaggle/input/tomato-leaf-disease/tomato/labels/val
/kaggle/input/tomato-leaf-disease/tomato/labels/train
/kaggle/input/tomato-leaf-disease/tomato/images
/kaggle/input/tomato-leaf-disease/tomato/images/val
/kaggle/input/tomato-leaf-disease/tomato/images/train


In [8]:
import os

for root, dirs, files in os.walk(TOMATO_ROOT):
    if "data.yaml" in files:
        print("FOUND:", os.path.join(root, "data.yaml"))

FOUND: /kaggle/input/tomato-leaf-disease/tomato/data.yaml


In [9]:
import yaml

yaml_path = "/kaggle/input/tomato-leaf-disease/tomato/data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

CLASS_NAMES = data["names"]

print("Number of classes:", len(CLASS_NAMES))

for i, cls in enumerate(CLASS_NAMES):
    print(i, "->", cls)

Number of classes: 10
0 -> Tomato__BacterialSpot
1 -> Tomato__EarlyBlight
2 -> Tomato__Healthy
3 -> Tomato__LateBlight
4 -> Tomato__LeafMold
5 -> Tomato__MosaicVirus
6 -> Tomato__SeptoriaLeafSpot
7 -> Tomato__SpiderMites
8 -> Tomato__TargetSpot
9 -> Tomato__YellowLeafCurlVirus


In [10]:
import os
import shutil
from tqdm import tqdm

ROOT = "/kaggle/input/tomato-leaf-disease/tomato"

IMG_TRAIN = os.path.join(ROOT, "images/train")
IMG_VAL = os.path.join(ROOT, "images/val")

LBL_TRAIN = os.path.join(ROOT, "labels/train")
LBL_VAL = os.path.join(ROOT, "labels/val")

OUT = "/kaggle/working/tomato_classification"

os.makedirs(OUT, exist_ok=True)

for split in ["train", "val"]:
    for cls in CLASS_NAMES:
        os.makedirs(
            os.path.join(OUT, split, cls),
            exist_ok=True
        )

def convert_split(img_dir, lbl_dir, split):

    images = os.listdir(img_dir)

    for img_file in tqdm(images):

        if not img_file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        label_file = os.path.splitext(img_file)[0] + ".txt"

        label_path = os.path.join(
            lbl_dir,
            label_file
        )

        if not os.path.exists(label_path):
            continue

        with open(label_path) as f:
            line = f.readline().strip()

        if line == "":
            continue

        class_id = int(line.split()[0])

        class_name = CLASS_NAMES[class_id]

        src = os.path.join(
            img_dir,
            img_file
        )

        dst = os.path.join(
            OUT,
            split,
            class_name,
            img_file
        )

        shutil.copy2(src, dst)

convert_split(
    IMG_TRAIN,
    LBL_TRAIN,
    "train"
)

convert_split(
    IMG_VAL,
    LBL_VAL,
    "val"
)

print("Conversion Complete")

100%|██████████| 3041/3041 [00:32<00:00, 94.77it/s]

Conversion Complete


In [11]:
import os

for cls in CLASS_NAMES:

    train_count = len(
        os.listdir(
            os.path.join(
                OUT,
                "train",
                cls
            )
        )
    )

    val_count = len(
        os.listdir(
            os.path.join(
                OUT,
                "val",
                cls
            )
        )
    )

    print(
        cls,
        "| Train:",
        train_count,
        "| Val:",
        val_count
    )

Tomato__BacterialSpot | Train: 1598 | Val: 400
Tomato__EarlyBlight | Train: 870 | Val: 218
Tomato__Healthy | Train: 1322 | Val: 332
Tomato__LateBlight | Train: 1599 | Val: 400
Tomato__LeafMold | Train: 834 | Val: 208
Tomato__MosaicVirus | Train: 342 | Val: 85
Tomato__SeptoriaLeafSpot | Train: 1536 | Val: 383
Tomato__SpiderMites | Train: 1343 | Val: 335
Tomato__TargetSpot | Train: 1124 | Val: 280
Tomato__YellowLeafCurlVirus | Train: 1600 | Val: 400


In [12]:
TOMATO_DATASET = "/kaggle/working/tomato_classification"

In [13]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    TOMATO_DATASET + "/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = image_dataset_from_directory(
    TOMATO_DATASET + "/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print(class_names)

Found 12168 files belonging to 10 classes.
Found 3041 files belonging to 10 classes.
['Tomato__BacterialSpot', 'Tomato__EarlyBlight', 'Tomato__Healthy', 'Tomato__LateBlight', 'Tomato__LeafMold', 'Tomato__MosaicVirus', 'Tomato__SeptoriaLeafSpot', 'Tomato__SpiderMites', 'Tomato__TargetSpot', 'Tomato__YellowLeafCurlVirus']


In [14]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(2000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [15]:
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

In [16]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [17]:
base_model = MobileNetV3Large(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [18]:
inputs = layers.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    10,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

In [19]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    )
]

In [21]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 631s 2s/step - accuracy: 0.6320 - loss: 1.1142 - val_accuracy: 0.8237 - val_loss: 0.6132
Epoch 2/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 635s 2s/step - accuracy: 0.8012 - loss: 0.6224 - val_accuracy: 0.8454 - val_loss: 0.5275
Epoch 3/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.8330 - loss: 0.5289 - val_accuracy: 0.8547 - val_loss: 0.4835
Epoch 4/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 616s 2s/step - accuracy: 0.8543 - loss: 0.4762 - val_accuracy: 0.8885 - val_loss: 0.4027
Epoch 5/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 589s 2s/step - accuracy: 0.8623 - loss: 0.4452 - val_accuracy: 0.8806 - val_loss: 0.4339
Epoch 6/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 615s 2s/step - accuracy: 0.8656 - loss: 0.4239 - val_accuracy: 0.8833 - val_loss: 0.4074
Epoch 7/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 630s 2s/step - accuracy: 0.8654 - loss: 0.4175 - val_accuracy: 0.8889 - val_loss: 0.3954
Epoch 8/15
381/381 ━━━━━━━━━━━━━━━━━━━━ 585s 2s/step - accuracy: 0.8699 - loss: 0.4051 - val_accu

In [22]:
print(max(history.history["accuracy"]))
print(max(history.history["val_accuracy"]))

0.8859302997589111
0.8967444896697998
